# Notebook 1: Lexical vs Semantic Search

A basic overview of lexical (keyword-based text searches) and semantic search.

Goal: Understand the difference between lexical search and semantic search. 

In [6]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

## Lexical vs Semantic — what is the difference?

**Lexical search** matches *the literal text* of the query — either as raw
substrings or as tokens.

The query "train" matches documents that contain the
literal substring `train` (e.g., "trains", "trained", etc.).  Under the hood each document is represented as a *sparse*
vector — one dimension per vocabulary token, almost all entries zero. The query `train` will only match posts that literally contain the substring
`train` (or `training`, `trainer`, etc. via tokenisation). It will *not* match a post
about a *high-speed rail line*, a *subway commute*, or a *locomotive*.

**Semantic search** matches *meaning*. 

Semantic search closes that gap by comparing *embeddings* of the query and document
instead of their tokens. Each document is projected onto a dense
vector (" word embedding") produced by a language model. Because the language model has learned associations
between synonyms, paraphrases, and related concepts during training, documents with related
meanings can be returned. A semantic search query for "train" can surface a post
about *the new high-speed rail line*, *subway compute* etc.


# What is an embedding?

An embedding is generated by a machine learning model that has been trained to
understand the relationships between words and phrases in a large corpus of
text. The resulting embedding is a vector of numbers that defines the position
of a piece of text relative to the corpus the model was trained on as a whole.


Link: Understanding Emeddings

# What is cosine similarity? 

Semantic search depends on cosine similarity, as the distance metric, to rank results.

Cosine similarity is the cosine of the angle between two vectors, normalized. Two
points pointing in the same direction from the origin score 1.0; perpendicular points
score 0.0.<br><br>

<img src="images/image_cosine_similarity.png" alt="Vector similarity illustration" style="width: 100%; max-width: 100%; height: auto;" />

Using the coordinates from the image, here is the end-to-end cosine similarity calculation for vectors $\vec{A}$ and $\vec{B}$.

Given vectors:

$$
\vec{A} = (1,2,3), \quad \vec{B} = (5,0,4)
$$

Step 1: Dot product

$$
\vec{A} \cdot \vec{B} = (1 \cdot 5) + (2 \cdot 0) + (3 \cdot 4) = 5 + 0 + 12 = 17
$$

Step 2: Magnitudes

$$
\lVert \vec{A} \rVert = \sqrt{1^2 + 2^2 + 3^2} = \sqrt{1+4+9} = \sqrt{14}
\approx 3.74
$$
$$
\lVert \vec{B} \rVert = \sqrt{5^2 + 0^2 + 4^2} = \sqrt{25+0+16} = \sqrt{41} \approx 6.40
$$

Step 3: Cosine similarity

$$
\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\lVert \vec{A} \rVert\,\lVert \vec{B} \rVert}
= \frac{17}{3.74 \times 6.40}
= \frac{17}{23.94}
\approx 0.71
$$

Final result: $0.71$.

Because the value is positive and fairly high, the two vectors point in a similar direction (angle about $44.8^\circ$).

## Cosine similarity in practice<br>


In [7]:
import numpy as np
import pandas as pd
from IPython.display import display, Math


# The formula for cosine similarity is:
display(Math(
    r"\text{Cosine Similarity} = "
    r"\frac{\sum_{i=1}^{n} A_i B_i}"
    r"{\sqrt{\sum_{i=1}^{n} A_i^2}\, \sqrt{\sum_{i=1}^{n} B_i^2}}"
))

# Using numpy, which can handle a vector of n-size efficiently.

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors: dot product / (|a| * |b|)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

A = np.array([1, 2, 3])
B = np.array([5,0,4])

# This is how we see the shape (number of dimensions) of the vectors.
print(f"Vector A dimensions: {A.shape}: {A}")
print(f"Vector B dimensions: {B.shape}: {B}")

# 1. Calculate the Dot Product
dot_product = np.dot(A, B)

# 2. Calculate the Magnitudes (Norms)
norm_A = np.linalg.norm(A)
norm_B = np.linalg.norm(B)

# 3. Calculate Cosine Similarity
similarity = dot_product / (norm_A * norm_B)

print(f"Cosine Similarity: {similarity:.2f}")


<IPython.core.display.Math object>

Vector A dimensions: (3,): [1 2 3]
Vector B dimensions: (3,): [5 0 4]
Cosine Similarity: 0.71


When we compare embeddings using cosine similarity, there are a few differences:

1. The vectors are typically high-dimensional (e.g., 384,512, 768, or even more dimensions) compared to the 3-dimensional example above.
2. The values in the vectors are usually floating-point numbers (e.g., 0.1, -0.2, 0.5) rather than integers.
3. Embedding models often return normalized vectors which have a length of 1, which means the cosine similarity can be directly computed as the dot product without needing to divide by the magnitudes.

# What is KNN (K-Nearest Neighbors)


In search retrieval, K-Nearest Neighbors (KNN) is the algorithm used to find the
most relevant items (like documents or images). 

When you use Cosine Similarity as "distance" the metric, you aren't looking for
the points that are physically "closest" in terms of distance (like miles); you
are looking for the points that point in the most similar direction.

How it works:

Every item in your database is stored with a "dense vector" (ie. an embedding)

The Query: When you search for something, your search term is also converted
into a vector.

The "K" in KNN: You decide how many results you want (e.g., \(K=5\)).

Finding Neighbors: The system calculates the Cosine Similarity between your
query vector and every vector in the database.

Sorting: It ranks the results from \(1.0\) (identical direction) down to \(-1.0\). The "K" items with the highest scores are your "Nearest Neighbors."


# Compare Search Results

## 1. Lexical search query
Run a lexical search over a handful of real posts using
`InMemoryKeywordSearcher` from `src/search.py`. This searcher does a simple
case-insensitive substring count — it is the simplest possible lexical search.

In [8]:
### Example of a basic lexical search.
import json

from src.config import REPO
from src.search import InMemoryKeywordSearcher

# load all the posts to be searched
with open(REPO / "sample_posts.json") as f:
    postdocs = json.load(f)

print(f"Loaded {len(postdocs)} posts. Showing first 3:")
for post in postdocs[:3]:
    print(f"  • {post['post_text'][:80]}…")

# Run a lexical search for the literal token "train"
searcher = InMemoryKeywordSearcher(postdocs)
results = searcher.search("train", top_k=5)

print(f"\nLexical search for 'train' returned {len(results)} matches:")
for result in results:
    print(f"  score={result['score']}  →  {result['post_text'][:100]}…")

Loaded 154 posts. Showing first 3:
  • 🐾 Today's Caturday is extra special because we're celebrating Whiskers' birthday…
  • 🐱 Kitty can be such a joy, but they sure do have their quirks too. Like preferri…
  • 👩‍👧‍👦 Catmom and Dad share equally in the responsibilities of raising a litter o…

Lexical search for 'train' returned 5 matches:
  score=1  →  Federal funding for California's High Speed Rail project has increased significantly over recent yea…
  score=1  →  Advocates argue that bullet trains are essential for reducing traffic jams and air pollution. With f…
  score=1  →  The transition from swimming in pools to open water can be daunting, but it's a rewarding experience…
  score=1  →  For those training for open water competitions in cold climates like Canada’s Niagara, gear choice b…
  score=1  →  Preparing for open water races involves more than just physical training; it's about understanding y…


Elasticsearch uses a more sophisticated lexical search, but the principle is
the same: it looks for literal token matches.

In [9]:
# Elasticsearch "BM25" (Best Matching 25), a common algorithm for ranking search results.
import inspect

from src.search import KeywordSearcher

print("── KeywordSearcher.search_similar_documents ──")
print(inspect.getsource(KeywordSearcher.search_similar_documents))

── KeywordSearcher.search_similar_documents ──
    def search_similar_documents(
        self, query: str, top_k: int = TOP_K_DEFAULT, filters: list[dict] | None = None
    ) -> list[dict]:
        """Run a BM25 match query against post_text and return ranked results."""
        body = {
            "size": top_k,
            "query": {"match": {"post_text": query}},
        }
        try:
            resp = self.client.search(index=self.index_name, body=body)
        except Exception as e:
            raise ConnectionError("Check that the docker container is running") from e
        results = [
            {
                "score": hit["_score"],
                **{k: v for k, v in hit["_source"].items() if k != "doc_embedding"},
            }
            for hit in resp["hits"]["hits"]
        ]
        return sorted(results, key=lambda result: result.get("score", 0.0), reverse=True)



## 2. Semantic search query

In [10]:
# First we need to add embeddings to all the documents we will search.
# We will walk through this in more detail in the next notebook.
# If this cell does not run for you, your environment may not have the necessary
# dependencies to run the embedding model. Ask for help!
# Depending on your computer, this may take a few minutes to run the first time as it downloads the model.

import logging
from solutions.preprocess import PreprocessingPipeline
# At the top of the cell you want to quiet:
logging.getLogger().setLevel(logging.ERROR)

PreprocessingPipeline().run()

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
### Example of a basic semantic search.
import json
from venv import logger

from src.config import OUTPUT # Path object
from src.search import InMemorySemanticSearcher
from sentence_transformers import SentenceTransformer
from src.data_models import PostDocument

with open(OUTPUT / "processed_posts.json") as f:
    data = json.load(f)
    postdocs = list(data.values())

print(f"Loaded {len(postdocs)} posts. Showing first 3:")
for post in postdocs[:3]:
    print(f"  • {post['post_text'][:80]}…")

# Generate a search embedding from just the text query

search_query = "train"

logger.info(f"Embedding query: {search_query}")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
search_embedding = embedding_model.encode(
    search_query,
    convert_to_numpy=True,
)

# Run a semantic search for the literal token "train"
searcher = InMemorySemanticSearcher(postdocs)
results = searcher.search_similar_documents(embedding=search_embedding, top_k=5)

print(f"\nSemantic search for 'train' returned {len(results)} matches:")
for result in results:
    print(f"  score={result['score']}  →  {result['post_text'][:100]}…")

Loaded 154 posts. Showing first 3:
  • 🐾 Today's Caturday is extra special because we're celebrating Whiskers' birthday…
  • 🐱 Kitty can be such a joy, but they sure do have their quirks too. Like preferri…
  • 👩‍👧‍👦 Catmom and Dad share equally in the responsibilities of raising a litter o…

Semantic search for 'train' returned 5 matches:
  score=0.39477667212486267  →  Let’s discuss SF to LA by 2035! 🚄 Infrastructure that connects our cities seamlessly, reducing trave…
  score=0.36579596996307373  →  Advocates argue that bullet trains are essential for reducing traffic jams and air pollution. With f…
  score=0.3608582615852356  →  High Speed Rail officials report that California's High Speed Rail is progressing despite recent fun…
  score=0.3330680727958679  →  For those training for open water competitions in cold climates like Canada’s Niagara, gear choice b…
  score=0.3190971612930298  →  With its potential to revolutionize how Californians travel from San Francisco to Los Angeles

Under the hood both the InMemorySemanticSearcher and ElasticSearch take dense
vectors and compute cosine similarity. 

In [12]:
import inspect

from src.search import SemanticSearcher

# This method uses Elasticsearch's scoring, but the search approach is the same.
print("── SemanticSearcher.search_similar_documents ──")
print(inspect.getsource(SemanticSearcher.search_similar_documents))

── SemanticSearcher.search_similar_documents ──
    def search_similar_documents(
        self,
        embedding: np.ndarray,
        top_k: int = TOP_K_DEFAULT,
        filters: list[dict] | None = None,
    ) -> list[dict]:
        """Vector search in Elasticsearch.

        - If top_k is set: return top_k docs ranked by cosine similarity.
        - If top_k is None: return all matching docs ranked by cosine similarity.
        """
        # ES requires size to be an integer; None means "return all up to the ES max"
        size = top_k if top_k is not None else 10_000

        filter_query = {"bool": {"filter": filters}} if filters else {"match_all": {}}
        body = {
            "size": size,
            "query": {
                "script_score": {
                    "query": filter_query,
                    "script": {
                        "source": "cosineSimilarity(params.query_vector, 'doc_embedding') + 1.0",
                        "params": {"query_vector": embedding

# Exercise: 

Get to know the data
Time: 3 minutes  

Before we move on to generating embeddings and building a topic model, it's helpful to get familiar with
the dataset.

Open [sample_posts.json](../sample_posts.json). 

```
REPO / sample_posts.json
```
- Use the search queries above to explore the dataset
- Identify three queries that would work in semantic search but not in lexical
  search.
- Identify abnormalities with the data that might make it harder to search for
  posts. 